# 3 · Samplers

WHISPER exposes four samplers behind one interface — `fit_<NAME>(lc, model, prior=..., ...)` → `SamplerResult`:

| sampler | family | notes |
|---|---|---|
| `fit_ABC` | likelihood-free rejection | simple, parallel, robust |
| `fit_ABC_SMC` | sequential ABC | importance-weighted, adaptive ε |
| `fit_MCMC` | exact likelihood (emcee) | gold-standard posterior |
| `fit_SNPE` | neural (sbi) | amortized; GPU-capable |

We fit synthetic data from a known truth so recovery is checkable.

In [1]:
import numpy as np
import whisper_labia as wp

MODEL = 'gaussian_rise'
truth = {'amplitude': 5.0, 't0': 8.0, 'sigma_rise': 3.0, 'tau_decay': 15.0}
t = np.linspace(0.1, 30, 60)
clean = wp.get_model(MODEL).predict(truth, t, None)
err = np.full_like(clean, 0.1)
obs = clean + np.random.default_rng(0).normal(0, err)
lc = wp.LightCurve(time=t, band=['r'] * len(t), flux=obs, flux_err=err, name='synthetic')
lc.n_points

60

In [2]:
prior = wp.Prior({'amplitude': wp.Uniform(0, 20), 't0': wp.Uniform(0, 20),
                  'sigma_rise': wp.Uniform(0.5, 10), 'tau_decay': wp.Uniform(1, 40)})

## Fit with each sampler

Budgets are kept small for speed. `SamplerResult` carries `.samples` (a DataFrame), `.summary` (median + credible intervals), `.best_params`, exact `.aic` / `.bic`, and `.runtime_s`.

In [3]:
res_abc = wp.fit_ABC(lc, MODEL, prior=prior, n_simulations=20000, quantile=0.01, seed=0)
res_smc = wp.fit_ABC_SMC(lc, MODEL, prior=prior, n_particles=400, n_rounds=4, seed=0)
res_mc  = wp.fit_MCMC(lc, MODEL, prior=prior, nsteps=2000, burnin=500, seed=0)
print('done')

done


In [4]:
import pandas as pd
rows = []
for name, r in [('ABC', res_abc), ('ABC-SMC', res_smc), ('MCMC', res_mc)]:
    s = r.summary
    rows.append(dict(sampler=name,
                     amplitude=round(s['amplitude']['median'], 2),
                     t0=round(s['t0']['median'], 2),
                     AIC=round(r.aic, 1), runtime_s=round(r.runtime_s, 1)))
pd.DataFrame(rows)

,sampler,amplitude,t0,AIC,runtime_s
0,ABC,4.30,8.40,195.0,0.4
1,ABC-SMC,4.08,8.57,656.3,0.9
2,MCMC,4.94,8.15,-117.6,3.0


Truth was `amplitude=5.0, t0=8.0`. All three recover it. `fit_SNPE` follows the same interface (add `device='cuda'` for GPU); it trains a neural posterior, so it costs more up front but can then condition on new data cheaply.

**Next:** [4 · Visualizing results](04_visualizing.ipynb).